# SpaceX Falcon 9 – Interactive Visual Analytics with Folium

In [1]:
import folium, pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from math import sin, cos, sqrt, atan2, radians
spacex_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
launch_sites_df = spacex_df.groupby('Launch Site', as_index=False).first()[['Launch Site','Lat','Long']]; launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [2]:
site_map = folium.Map(location=[29.559684888503615, -95.0830971930759], zoom_start=5)
for _, r in launch_sites_df.iterrows():
    folium.Circle([r.Lat, r.Long], radius=1000, color='#d35400', fill=True).add_child(folium.Popup(r['Launch Site'])).add_to(site_map)
    folium.map.Marker([r.Lat, r.Long], icon=folium.DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html=f"<div style='font-size:12;color:#d35400;'><b>{r['Launch Site']}</b></div>")).add_to(site_map)
site_map

In [3]:
marker_cluster = MarkerCluster().add_to(site_map)
for _, r in spacex_df.iterrows():
    folium.Marker([r.Lat, r.Long], icon=folium.Icon(color='white', icon_color='green' if r['class']==1 else 'red')).add_to(marker_cluster)
site_map.add_child(MousePosition(position='topright', separator=' Long: ', prefix='Lat:'))
site_map

In [4]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0; lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))
site = (28.56342, -80.57674)
for name, pt in {'Coastline': (28.56367, -80.56794), 'Railway': (28.57206, -80.58528), 'Highway': (28.56321, -80.57107), 'City': (28.10473, -80.64531)}.items():
    d = calculate_distance(*site, *pt); print(name, round(d, 2), 'km')
    folium.Marker(pt, icon=folium.DivIcon(html=f"<div style='font-size:12;color:#d35400;'><b>{d:.2f} KM</b></div>")).add_to(site_map)
    folium.PolyLine([site, pt], weight=1).add_to(site_map)
site_map.save('launch_sites_map.html'); site_map

Coastline 0.86 km
Railway 1.27 km
Highway 0.55 km
City 51.46 km
